In [ ]:

from wordfreq import word_frequency
from wordfreq import get_frequency_dict
import numpy as np
from matplotlib import pyplot as plt

# Load the datafile with all the wordle words
words = np.genfromtxt('valid-wordle-words.txt', delimiter='\n', dtype=str)  # all the possible guesses
word_frequencies = [word_frequency(words[i], 'en') for i in range(len(words))]  # frequency of each guess

dic = np.vstack((words, word_frequencies))     # 2 x 12972 array: word, frequency
i = np.argsort(word_frequencies)[::-1]
dic = dic[:, i]                                # sort by frequency, descending
ind_0 = np.where(dic[1] == '0.0')
dic[1, ind_0] = '1e-10'                        # set freq 1e-10 for all words with freq 0

dic_2309 = np.delete(dic, slice(2309, 12972), 1)  # top 2309 most common words
sol_ov = np.genfromtxt('wordle-nyt-answers-alphabetical.txt', delimiter='\n', dtype=str)  # set of solutions
word_frequencies_sol = [word_frequency(sol_ov[i], 'en') for i in range(len(sol_ov))]

# None of the words in the set of solutions has frequency 0
intersection = np.intersect1d(dic_2309[0], sol_ov)  # words in both the top 2309 and the solution set
size_inter_2309 = np.shape(intersection)[0]
print("The {:.5}% of the words in the original set of solutions are in the top {} of most common words."
      .format(100 * (np.shape(intersection)[0]) / (np.shape(sol_ov)[0]), np.shape(dic_2309)[1]))
print("Then, the {:.5}% of the words in the original set of solutions are not.\n"
      .format(100 - 100 * (np.shape(intersection)[0]) / (np.shape(sol_ov)[0])))


# Distribution of the set of all possible guesses by frequency interval.
# Each position in the list represents an interval (10^-(k+1), 10^-k].
print("The set of all possible guesses is distributed as follows: \n")
dist_guesses = [0 for i in range(10)]
ind_0 = [i for i in word_frequencies if i == 0.0]
for k in range(1, 11):
    ind = [i for i in word_frequencies if i <= 10 ** -(k) and i > 10 ** (-(k + 1))]
    dist_guesses[k - 1] = len(ind)
dist_guesses[9] = len(ind_0)
print(dist_guesses)


# Distribution of the set of solutions by frequency interval.
print("The set of solutions is distributed as follows: \n")
dist_sol = [0 for i in range(10)]
ind_sol_0 = [i for i in word_frequencies_sol if i == 0.0]
percentages_list = [0] * 10
for k in range(1, 11):
    ind_sol = [i for i in word_frequencies_sol if i <= 10 ** -(k) and i > 10 ** (-(k + 1))]
    dist_sol[k - 1] = len(ind_sol)
    if dist_guesses[k - 1] != 0:
        percentages_list[k - 1] = len(ind_sol) / (dist_guesses[k - 1])
dist_sol[9] = len(ind_sol_0)
print(dist_sol)
print("The percentage of words added from each interval is given by the following list:")
print(percentages_list)


# All guesses: word index vs frequency (not normalized)
xpoints = np.arange(start=0, stop=len(words), step=1)
markers_on = [ind for ind, i in enumerate(dic[0]) if i in sol_ov]
freq_setguesses = [eval(i) for i in dic[1]]
print(np.sum(freq_setguesses))

plt.plot(xpoints, freq_setguesses)
plt.yscale('log')
plt.title('Distribution frequencies (set of all possible guesses)')
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.show()


# Normalize the word frequencies so the area under the curve is 1
Sum = np.sum(freq_setguesses)
freq_setguesses = freq_setguesses * 1 / Sum
word_frequencies_sol = word_frequencies_sol * 1 / Sum
Area_sol = np.sum(word_frequencies_sol)  # contribution of the solution words to the normalized area

# All guesses: word index vs frequency (normalized), with solution words marked
plt.plot(xpoints, freq_setguesses, '-bx', markevery=markers_on, label='line with select markers')
plt.yscale('log')
plt.title('Distribution frequencies (set of all possible guesses)')
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.show()
print("The area under the curve is 1 and the area of the elements in the set of solutions is {:.5}"
      .format(Area_sol))


# Top-2309-most-common guesses vs the solution set (not normalized)
xpoints = np.arange(start=0, stop=len(dic_2309[1]), step=1)
freq_setsol = [eval(i) for i in dic_2309[1]]
freq_inter = [eval(i) for i in dic[1][0:size_inter_2309 - 1]]
plt.plot(xpoints, freq_setsol, 'b', np.arange(start=0, stop=size_inter_2309 - 1, step=1), freq_inter, 'r')
plt.yscale('log')
plt.title('Distribution frequencies (set of solutions)')
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.show()


# Same comparison, normalized, with the shared area shaded
Sum = np.sum(freq_setsol)
freq_setsol = freq_setsol * 1 / Sum
freq_inter = freq_inter * 1 / Sum
Area = np.sum(freq_inter)

plt.plot(xpoints, freq_setsol, 'b', np.arange(start=0, stop=size_inter_2309 - 1, step=1), freq_inter, 'r')
plt.yscale('log')
plt.fill_between(np.arange(start=0, stop=size_inter_2309 - 1, step=1), freq_inter, alpha=0.7)
# the blue-shaded region represents the area covered by the top-2309 words
plt.title('Distribution frequencies (set of solutions)')
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.show()
print("The area under the curve is 1 and the area of the blue region is {:.5}".format(Area))
